# Market Regime Detection — Walkthrough

This notebook narrates how the detector works: load prices, engineer
features, fit four models (a volatility baseline, KMeans, a Gaussian
Mixture, and a **Gaussian HMM**), and compare what they find.

> **Research prototype, not a trading system.** See the README's
> *Limitations* section for why. The goal is to show the modelling and
> engineering, and to be honest about the gap to production.

If Yahoo Finance is unreachable, the data loader falls back to a
deterministic synthetic series with three baked-in regimes, so this
notebook runs anywhere.

In [1]:
import sys; sys.path.insert(0, '../src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from regime_detector import data, features, evaluate, plots
from regime_detector.models import baseline, clustering, hmm

pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

## 1. Load prices

Daily adjusted close for SPY (2005–2024). Cached locally after the first
download.

In [2]:
prices = data.load_prices('SPY', start='2005-01-01', end='2024-01-01')
prices.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2023-12-26,140.307,140.325,137.743,138.029,17367456
2023-12-27,138.029,140.015,137.756,139.493,45272989
2023-12-28,139.493,141.140,139.051,140.641,30433040
2023-12-29,140.641,140.847,136.887,137.071,21892466
2024-01-01,137.071,137.522,136.194,136.351,28445006


## 2. Engineer features

From the close price we derive returns, rolling realized & downside
volatility, drawdown-from-peak, and a rolling return z-score — the signals
a regime lives in.

In [3]:
feats = features.build_features(prices)
X = features.model_matrix(feats)   # standardized model input
feats.describe()

,log_return,realized_vol,downside_vol,drawdown,return_zscore
count,"4,935.000","4,935.000","4,935.000","4,935.000","4,935.000"
mean,0.000,0.148,0.085,-0.441,-0.002
std,0.010,0.059,0.046,0.178,0.991
min,-0.079,0.062,0.023,-0.651,-3.762
25%,-0.005,0.109,0.057,-0.574,-0.664
50%,0.000,0.128,0.071,-0.517,0.001
75%,0.006,0.172,0.099,-0.358,0.642
max,0.062,0.428,0.301,0.000,3.628


In [4]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
prices.loc[feats.index, 'Close'].plot(ax=ax[0], color='black', lw=1)
ax[0].set_title('SPY close'); ax[0].set_ylabel('Price')
feats['realized_vol'].plot(ax=ax[1], color='#d7191c', lw=1)
ax[1].set_title('Realized volatility (annualized)'); ax[1].set_ylabel('Vol')
plt.tight_layout()

## 3. Fit the four models

Each returns one integer regime label per day, relabeled so regime 0 is
always the calmest state.

In [5]:
labels = {
    'Threshold baseline': baseline.fit_predict(feats, n_regimes=3),
    'KMeans': clustering.kmeans_fit_predict(feats, X, n_regimes=3),
    'GMM': clustering.gmm_fit_predict(feats, X, n_regimes=3),
    'HMM': hmm.fit_predict(feats, X, n_regimes=3),
}

## 4. Which model produces stable regimes?

A good regime detector shouldn't flip states every few days. The HMM,
which learns a transition matrix, should be far stickier than the
memoryless baseline.

In [6]:
evaluate.stability_report(labels)

,n_regimes,n_switches,avg_run_length_days
model,,,
Threshold baseline,3,414,11.892
KMeans,3,458,10.752
GMM,3,307,16.023
HMM,3,74,65.800


## 5. The headline chart

Close price shaded by regime, one panel per model. The baseline is
confetti; the HMM finds clean blocks.

In [7]:
plots.plot_model_comparison(prices, feats, labels)
plt.show()

## 6. What do the HMM's regimes mean economically?

Per-regime annualized return, volatility, Sharpe, and drawdown. Regime 0
should look calm and profitable; the top regime should look like a
crisis.

In [8]:
summary = evaluate.regime_summary(feats, labels['HMM'])
summary

,n_days,pct_time,ann_return,ann_vol,sharpe,avg_drawdown,worst_drawdown
regime,,,,,,,
0,2225,0.451,0.078,0.118,0.663,-0.550,-0.635
1,1344,0.272,0.084,0.127,0.667,-0.209,-0.451
2,1366,0.277,-0.132,0.231,-0.571,-0.493,-0.651


## 7. The learned transition matrix

The HMM's transition matrix shows how sticky each hidden state is (the
diagonal) — the source of its persistence.

In [9]:
det = hmm.HMMRegimeDetector(n_regimes=3)
det.fit_predict(feats, X)
print('Transition matrix (raw states):')
print(np.round(det.transition_matrix, 3))
print('\nExpected regime duration (days):', np.round(det.expected_durations(), 1))

Transition matrix (raw states):
[[0.974 0.008 0.018]
 [0.01  0.99  0.   ]
 [0.01  0.001 0.989]]

Expected regime duration (days): [ 37.9 101.3  89.3]


## 8. Walk-forward: what would you actually have known?

Everything above uses hindsight — the models see the whole history. A live system doesn't. Here we refit the HMM on an **expanding window** and label each day using only data up to that day (online inference), then compare against the hindsight labels to measure the detection **lag**.

> This takes ~40s on real data — it refits the HMM many times.

In [10]:
from regime_detector import walkforward as wf

online = wf.walk_forward_hmm(feats, n_regimes=3)
hindsight = labels['HMM']
print('Online vs hindsight agreement:', round(wf.agreement_rate(online, hindsight), 3))

Online vs hindsight agreement: 0.651


In [11]:
crisis = int(max(hindsight))
wf.detection_lag(online, hindsight, target_regime=crisis)

{'target_regime': 2,
 'n_events': 36,
 'n_detected': 30,
 'n_missed': 6,
 'mean_lag_days': 0.8666666666666667,
 'median_lag_days': 0.0}

The online panel below is noisier and starts later (warm-up) than the hindsight panel — that difference is exactly what a real-time system pays for not being able to see the future.

In [12]:
plots.plot_online_vs_hindsight(prices, feats, online, hindsight)
plt.show()

## Takeaways

- The **HMM** produces the most persistent, economically-distinct regimes
  because it models transitions — the baseline and clustering methods,
  which treat each day independently, flicker.
- Detected regimes line up with known market history (2008, 2020) when
  run on real data.
- **This is not tradeable as-is** — it uses full-sample fitting and
  smoothed (non-causal) inference. See the README for the full list and
  the walk-forward next step.